### Minizinc 2

[Video](https://youtu.be/P17s9UINCs4)

#### Teilmenge wählen (Knapsack)

5 verschiedene Moves können trainiert werden. Jedes Training dauert eine gewisse Zeit und liefert eine bestimmte Power.

<img src='turban.png' width='400'>

Eine Set-Variable als Decision-Variable wählt eine der möglichen Teilmengen einer Menge.

In [ ]:
int: N = 12;
enum MOVES = {lila,gelb,gruen,blau,rot};
array[MOVES] of int: power = [6,8,5,3,4];
array[MOVES] of int: duration = [4,5,3,2,3];

var set of MOVES: wahl;

constraint sum(m in wahl) (duration[m]) <= N;
solve maximize sum(m in wahl)(power[m]);

In [ ]:
wahl = {lila, gelb, gruen};
_objective = 19;
----------
==========
Finished in 163msec.

Minizinc unterstützt einige Set-Operationen:

    in (membership e.g. x in s)
    subset, superset
    intersect (intersection)
    union
    card (cardinality)
    diff (set difference, e.g x diff y = x \ y )
    symdiff (symmetric difference)

#### Bagua

<img src='bagua1.png' width='400'>

<img src='bagua3.png' width='400'>

<img src='bagua.png' width='400'>

<img src='bagua2.png' width='400'>



In [ ]:
int: anzahl = 10;
set of int: PUNKTE = 1..anzahl;  
array[PUNKTE] of int: schaden = [10,8,4,2,6,9,5,3,8,10];
enum SYMBOLE = {A,B,C,D,E,F,G,H};
array[SYMBOLE] of set of PUNKTE: gruppen = [{1,4,6},{1,2,6,7},{1,3,6,8},{1,2,3},{2,9,10},{5,6,8,10},{7,8,10},{1,3,5}];

var set of PUNKTE: attacks;

constraint forall(s in SYMBOLE)(card(attacks intersect gruppen[s]) <= 1);

solve maximize (sum(p in attacks)(schaden[p]));

#### Teilmengen fester Größe

Wenn die Lösung genau 3 Elemente haben soll, können wir einen zusätzlichen constraint einfügen:

    constraint(card(attacks)) = 3;

Wenn die Größe der gesuchten Teilmenge im Vergleich zur gesamten Menge relativ klein ist, ist meist die Modellierung mit einen array effektiver. Mit einem constraint sorgen wir dafür, dass eine Lösung des Modells genau einer Lösung des Problems entspricht. 

In [ ]:
int: N = 3;        % Anzahl attack-Punkte
int: anzahl = 10;
set of int: PUNKTE = 1..anzahl;
array[PUNKTE] of int: schaden = [10,8,4,2,6,9,5,3,8,10];
enum SYMBOLE = {A,B,C,D,E,F,G,H};
array[SYMBOLE] of set of PUNKTE: gruppen = [{1,4,6},{1,2,6,7},{1,3,6,8},{1,2,3},{2,9,10},{5,6,8,10},{7,8,10},{1,3,5}];
 
array[1..N] of var PUNKTE: attacks;
constraint forall(i in 1..N-1) (attacks[i] < attacks[i+1]);      % eine Menge der Größe N wird genau durch 1 array repräsentiert

constraint forall(s in SYMBOLE)(sum(i in 1..N)(attacks[i] in gruppen[s]) <= 1);

solve maximize (sum(p in attacks)(schaden[p]));

#### Teilmengen beschränkter Größe

Wenn die Lösung <= 3 Elemente haben soll, können wir den constraint einfügen:

    constraint(card(attacks)) <= 3;

Wenn die Größe der gesuchten Teilmenge im Vergleich zur gesamten Menge relativ klein ist, ist meist die Modellierung mit einen array wieder effektiver. Unser array hat wieder die Länge 3, wir fügen aber ein zusätzlichens Element 0 ein, das signalisiert, dass nichts gewählt wurde

    [2,0,0]   bedeutet die Menge {2}
    [4,3,0]   bedeutet die Menge {4,3}

Wir sortieren diesmal abwärts, damit die Nullen am Ende stehen, Nullen dürfen sich wiederholen, Nicht-Nullen müssen abwärts sortiert sein. 
Um dies zu erreichen gibt es einen cleveren constraint.

In [ ]:
int: N = 3;         
int: anzahl = 10;
set of int: PUNKTE = 1..anzahl;
set of int: PUNKTEX = {0} union PUNKTE;
array[PUNKTE] of int: schaden = [10,8,4,2,6,9,5,3,8,10];
enum SYMBOLE = {A,B,C,D,E,F,G,H};
array[SYMBOLE] of set of PUNKTE: gruppen = [{1,4,6},{1,2,6,7},{1,3,6,8},{1,2,3},{2,9,10},{5,6,8,10},{7,8,10},{1,3,5}];
 
array[1..N] of var PUNKTEX: attacks;

constraint forall(i in 1..N-1) (attacks[i] >= (attacks[i]!=0) + attacks[i+1]);   % keine Wiederholung für Elemente, die nicht 0 sind.

constraint forall(s in SYMBOLE)(sum(i in 1..N)(attacks[i] in gruppen[s]) <= 1);

solve maximize (sum(p in attacks where p> 0)(schaden[p]));     % es gibt kein schaden mit Index p

#### Pure Assignment Problem

Wir haben 3 Angreifer, die können mit unterschiedlichem Ergebnis 5 Punkte angreifen. Keine Punkte kann von zwei Angreifern angegriffen werden.
Welche Punkte müssen wir den Angreifern zuordnen um den größten Schaden zu bewirken?

Wir wollen eine Funktion, die jedem Element einer Domain (DOM = 3 Angreifer) ein Element der Codomain (COD = Angriffspunkte) zuordnet.

Der global constraint *alldifferent* sorgt dafür, dass die Funktion injektiv ist. 

<img src='assign1.png' width='500'>

In [ ]:
include "alldifferent.mzn";

enum HERO = {A, B, C};
enum SPOT = {S1, S2, S3, S4, S5};
 
array[HERO,SPOT] of int: damage = [|7, 1, 3, 4, 6| 8, 2, 5, 1, 4| 4, 3, 7, 2, 5|];
array[HERO] of var SPOT: pos;

constraint alldifferent(pos);

var int: tDamages = sum(h in HERO)(damage[h,pos[h]]);
solve maximize tDamages;

output ["\(h): \(pos[h])\n" | h in HERO] ++ ["Total Damages: \(tDamages)"];

#### Assignment2: Prison

<img src='prison.png' width='500'>

<img src='prison2.png' width='300'>

<img src='prison3.png' width='300'>

<img src='prison4.png' width='300'>

In [ ]:
include "alldifferent.mzn";

enum PRISONER =  {P1, P2, P3, P4, P5, P6, P7, P8, P9, P10};

int: n = 4;  % number of row in prison grid 
set of int: ROW = 1..n;

int: m = 6;  % number of columns in prison grid
set of int: COL = 1..m;

array[ROW,COL] of int: cost = [|2, 4, 3, 5, 1, 6 | 6, 2, 3, 1, 5, 4 | 3, 5, 6, 4, 2, 1 | 1, 2, 6, 4, 5, 3|];
set of PRISONER: danger = {P1, P4, P8};
set of PRISONER: female = {P1, P6, P7, P10};
set of PRISONER: male = PRISONER diff female;

array[PRISONER] of var ROW: r;
array[PRISONER] of var COL: c;

constraint alldifferent([(r[p]-1)*m + c[p] | p in PRISONER]);

constraint forall (p in female) (r[p] <= n div 2);
constraint forall (p in male) (r[p] > n div 2);

constraint forall (d in danger, p in PRISONER where d != p) ((abs(r[d] - r[p]) + abs(c[d] - c[p])) > 1);

var int: tCost = sum (p in PRISONER) (cost[r[p],c[p]]);
solve minimize tCost;

output["Female: "++show(p)++" in cell["++show(r[p])++","++show(c[p])++"]\n" | 
      p in female]
  ++ ["\n"]++["Male: "++show(p)++" in cell["++show(r[p])++","++show(c[p])++"]\n" | 
      p in male]
  ++ ["\nTotal Cost: ",show(tCost)];

#### Dienstplanprobleme  (Rostering Problems)

Für die soldiers sollen Dienstpläne für die Wachen erstellt werden. Es gelten folgende Regeln:

1. Genau 3 Soldaten für die Nachtwache
2. 1 oder 2 Soldaten für die Abendwache
3. Keine drei Nachtwachen hintereinander
4. Keine Nachtwache direkt nach einer Abendwache


<img src='shift.png' width='400'>

<img src='shift1.png' width='400'>

In [ ]:
enum SOLDIER = {S1,S2,S3,S4,S5,S6};
enum SHIFT = { OFF, EVE, NIGHT };

int: nDays = 8; 
set of int: DAY = 1..nDays;

int: o = 3;
int: l = 1;
int: u = 2;

array[SOLDIER,DAY] of var SHIFT: roster;

constraint forall(d in 1..(nDays-2), s in SOLDIER)((roster[s,d] = NIGHT) /\ (roster[s,d+1] = NIGHT) -> (roster[s,d+2] != NIGHT));
constraint forall(d in 1..(nDays-1), s in SOLDIER)((roster[s,d] = EVE) -> (roster[s,d+1] != NIGHT));

constraint forall(d in DAY)(sum (s in SOLDIER)((roster[s,d] = NIGHT)) = o);

constraint forall(d in DAY)(sum (s in SOLDIER)((roster[s,d] = EVE)) >= l);
constraint forall(d in DAY)(sum (s in SOLDIER)((roster[s,d] = EVE)) <= u);

var int: tOnEve = sum(d in DAY)(sum(s in SOLDIER)(roster[s,d] = EVE));
solve maximize (tOnEve);

output ["Soldier \(s) on Day \(d) takes the \(roster[s,d]) shift\n"
        ++ if s == max(SOLDIER) then "\n" else "" endif | d in DAY, s in SOLDIER]++[show(tOnEve)];

Boolesche Operatoren:

    /\ und
    \/ oder
    -> wenn-dann
    <-> genau-dann-wenn
    not negation

Für die Laufzeit ist es günstig, mehrfach verwendete Ausdrücke in Zwischenvariablen zu speichern. Das gilt sowohl für Parameter als auch für Decision Variablen.

In [ ]:
enum SOLDIER = {S1,S2,S3,S4,S5,S6};
enum SHIFT = { OFF, EVE, NIGHT };

int: nDays = 8; 
set of int: DAY = 1..nDays;

int: o = 3;
int: l = 1;
int: u = 2;

array[SOLDIER, DAY] of var SHIFT: roster;

constraint forall (d in 1..(nDays-2), s in SOLDIER) ((roster[s,d] = NIGHT) /\ (roster[s,d+1] = NIGHT) -> (roster[s,d+2] != NIGHT));
constraint forall (d in 1..(nDays-1), s in SOLDIER) ((roster[s,d] = EVE) -> (roster[s,d+1] != NIGHT));

constraint forall (d in DAY) (sum (s in SOLDIER) ((roster[s,d] = NIGHT)) = o);

array[DAY] of var l..u: onEve;
constraint onEve = [sum (s in SOLDIER) (roster[s,d]=EVE) | d in DAY];

solve maximize sum(onEve);

output ["Soldier \(s) on Day \(d) takes the \(roster[s,d]) shift\n" 
        ++ if s == max(SOLDIER) then "\(onEve[d])\n" else "" endif | d in DAY, s in SOLDIER]
        ++[show(sum(onEve))];

In [ ]:
Soldier S1 on Day 1 takes the EVE shift
Soldier S2 on Day 1 takes the NIGHT shift
Soldier S3 on Day 1 takes the OFF shift
Soldier S4 on Day 1 takes the NIGHT shift
Soldier S5 on Day 1 takes the NIGHT shift
Soldier S6 on Day 1 takes the EVE shift
2
Soldier S1 on Day 2 takes the OFF shift
Soldier S2 on Day 2 takes the NIGHT shift
Soldier S3 on Day 2 takes the NIGHT shift
Soldier S4 on Day 2 takes the OFF shift
Soldier S5 on Day 2 takes the NIGHT shift
Soldier S6 on Day 2 takes the EVE shift
1
Soldier S1 on Day 3 takes the NIGHT shift
Soldier S2 on Day 3 takes the EVE shift
Soldier S3 on Day 3 takes the NIGHT shift
Soldier S4 on Day 3 takes the NIGHT shift
Soldier S5 on Day 3 takes the OFF shift
Soldier S6 on Day 3 takes the EVE shift
2
Soldier S1 on Day 4 takes the NIGHT shift
Soldier S2 on Day 4 takes the EVE shift
Soldier S3 on Day 4 takes the OFF shift
Soldier S4 on Day 4 takes the NIGHT shift
Soldier S5 on Day 4 takes the NIGHT shift
Soldier S6 on Day 4 takes the OFF shift
1
Soldier S1 on Day 5 takes the OFF shift
Soldier S2 on Day 5 takes the EVE shift
Soldier S3 on Day 5 takes the NIGHT shift
Soldier S4 on Day 5 takes the EVE shift
Soldier S5 on Day 5 takes the NIGHT shift
Soldier S6 on Day 5 takes the NIGHT shift
2
Soldier S1 on Day 6 takes the NIGHT shift
Soldier S2 on Day 6 takes the OFF shift
Soldier S3 on Day 6 takes the NIGHT shift
Soldier S4 on Day 6 takes the EVE shift
Soldier S5 on Day 6 takes the OFF shift
Soldier S6 on Day 6 takes the NIGHT shift
1
Soldier S1 on Day 7 takes the NIGHT shift
Soldier S2 on Day 7 takes the NIGHT shift
Soldier S3 on Day 7 takes the EVE shift
Soldier S4 on Day 7 takes the EVE shift
Soldier S5 on Day 7 takes the NIGHT shift
Soldier S6 on Day 7 takes the OFF shift
2
Soldier S1 on Day 8 takes the EVE shift
Soldier S2 on Day 8 takes the NIGHT shift
Soldier S3 on Day 8 takes the EVE shift
Soldier S4 on Day 8 takes the OFF shift
Soldier S5 on Day 8 takes the NIGHT shift
Soldier S6 on Day 8 takes the NIGHT shift
2
13
----------
==========
Finished in 7s 270msec.

Mit einem global_cardinality constraint können wir die Laufzeit weiter verbessern.


    global_cardinality(x, v, c)     % x: data, v: values, c: counts
    global_cardinality([1,3,1,2,5], [1,2], [2,1])

In [ ]:
include "globals.mzn";
enum SOLDIER = {S1,S2,S3,S4,S5,S6};
enum SHIFT = { OFF, EVE, NIGHT };

int: nDays = 8; 
set of int: DAY = 1..nDays;

int: o = 3;
int: l = 1;
int: u = 2;

array[SOLDIER, DAY] of var SHIFT: roster;

constraint forall (d in 1..(nDays-2), s in SOLDIER) ((roster[s,d] = NIGHT) /\ (roster[s,d+1] = NIGHT) -> (roster[s,d+2] != NIGHT));
constraint forall (d in 1..(nDays-1), s in SOLDIER) ((roster[s,d] = EVE) -> (roster[s,d+1] != NIGHT));

array[DAY] of var l..u: onEve;
constraint forall (d in DAY) (global_cardinality([roster[s,d] | s in SOLDIER], [NIGHT, EVE], [o, onEve[d]]));

solve maximize sum(onEve);


output ["Soldier \(s) on Day \(d) takes the \(roster[s,d]) shift\n" 
        ++ if s == max(SOLDIER) then "\(onEve[d])\n" else "" endif | d in DAY, s in SOLDIER]++[show(sum(onEve))];

Varianten des global_cardinality contraints

    global_cardinality_closed(x, v, c)               % alle x-Werte auch in v
    global_cardinality_low_up(x, v, lo, hi)          % counts zwischen lo und hi 
    global_cardinality_low_up_closed(x, v, lo, hi)   % counts zwischen lo und hi, alle x-Werte auch in v